In [41]:
!pip install selenium webdriver-manager beautifulsoup4 requests

In [42]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import requests
import time
from pathlib import Path
from selenium.common.exceptions import NoSuchElementException



In [43]:
SIGNS = [
    'เข้าใจ', 'ไม่', 'ขอบคุณ', 'คนหูหนวก', 'คำถาม',
    'คุณ', 'ฉัน', 'ช่วย', 'ดี', 'ต้องการ',
    'ถาม', 'บอก', 'พูด', 'รู้', 'สวัสดี'
]
BASE_URL   = 'https://www.th-sl.com'
SEARCH_URL = 'https://www.th-sl.com/?s='
SAVE_DIR   = Path('D:/ThSL_Project/dataset/raw_videos')

In [44]:
options = webdriver.ChromeOptions()
# options.add_argument('--headless')  # uncomment to run without browser window

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install())
)
print('Driver ready')

Driver ready


In [45]:
def find_sign_url(sign: str) -> str:
    
    page = 1
    base_url = "https://www.th-sl.com/" 
    
    while True:

        if page == 1:
            current_url = f"{base_url}?s={sign}"
        else:
            current_url = f"{base_url}page/{page}/?s={sign}"

        driver.get(current_url)
        
        if page > 1 and (driver.current_url == base_url or "?page=1" in driver.current_url):
            print("Redirected back to the beginning. Sign not found.")
            break
        
        try:
            elements = driver.find_elements(By.XPATH, f"//a[contains(normalize-space(), '{sign}') and contains(@href, '/word/')]")
            if elements:
                return [el.get_attribute("href") for el in elements]
            
        except NoSuchElementException:
            print(f"'{sign}' not found on page {page}.")
        
        page += 1
        
    return "URL not found"

In [46]:
def get_video_url(page_url: str) -> str:
    """
    Visits a sign page and extracts the direct .mp4 video URL.
    """
    driver.get(page_url)
    time.sleep(3)

    soup   = BeautifulSoup(driver.page_source, 'html.parser')
    source = soup.find('source')

    if source and source.get('src'):
        return source['src']

    return ''

In [47]:
def download_video(video_url: str, save_path: Path) -> bool:
    """
    Downloads a video from URL and saves to save_path.
    Returns True if successful.
    """
    try:
        response = requests.get(video_url, stream=True, timeout=30)
        if response.status_code == 200:
            save_path.parent.mkdir(parents=True, exist_ok=True)
            with open(str(save_path), 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            return True
    except Exception as e:
        print(f'Download error: {e}')
    return False


In [48]:
def scrape_all():
    """
    Loops through all signs, finds video URL, downloads to correct folder.
    """
    results = {}

    for sign in SIGNS:
        print(f'\nProcessing: {sign}')
        save_path = SAVE_DIR / sign / 'scraped_001.mp4'

        # Skip if already downloaded
        if save_path.exists():
            print(f'   Already downloaded')
            results[sign] = 'skipped'
            continue

        # Find sign page URL
        page_urls = find_sign_url(sign)
        for i, page_url in enumerate(page_urls):
            save_path = SAVE_DIR / sign / f'scraped_{i+1:03d}.mp4'
            video_url = get_video_url(page_url)
            success = download_video(video_url, save_path)
        if not page_url:
            print(f'  Sign not found on website')
            results[sign] = 'not found'
            continue

        # Get video URL
        video_url = get_video_url(page_url)
        if not video_url:
            print(f'   No video found on page')
            results[sign] = 'no video'
            continue

        # Download
        success = download_video(video_url, save_path)
        if success:
            print(f'   Downloaded → {save_path.name}')
            results[sign] = 'success'
        else:
            print(f'   Download failed')
            results[sign] = 'failed'

        time.sleep(2)  # be polite to the server

    # Summary
    print('\n' + '='*40)
    for sign, status in results.items():
        icon = '✅' if status == 'success' else '⏭️ ' if status == 'skipped' else '❌'
        print(f'  {icon} {sign:12s}: {status}')

    driver.quit()

In [49]:
scrape_all()


Processing: เข้าใจ
   Already downloaded

Processing: ไม่
   Already downloaded

Processing: ขอบคุณ
   Already downloaded

Processing: คนหูหนวก
   Already downloaded

Processing: คำถาม
   Already downloaded

Processing: คุณ
   Already downloaded

Processing: ฉัน
   Already downloaded

Processing: ช่วย
   Already downloaded

Processing: ดี
   Already downloaded

Processing: ต้องการ
   Already downloaded

Processing: ถาม
   Already downloaded

Processing: บอก
   Already downloaded

Processing: พูด
   Already downloaded

Processing: รู้
   Already downloaded

Processing: สวัสดี
   Downloaded → scraped_002.mp4

  ⏭️  เข้าใจ      : skipped
  ⏭️  ไม่         : skipped
  ⏭️  ขอบคุณ      : skipped
  ⏭️  คนหูหนวก    : skipped
  ⏭️  คำถาม       : skipped
  ⏭️  คุณ         : skipped
  ⏭️  ฉัน         : skipped
  ⏭️  ช่วย        : skipped
  ⏭️  ดี          : skipped
  ⏭️  ต้องการ     : skipped
  ⏭️  ถาม         : skipped
  ⏭️  บอก         : skipped
  ⏭️  พูด         : skipped
  ⏭️  รู้         : 